In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel, AdamW
import numpy as np

# ---------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN = 128
BATCH_SIZE = 2
EPOCHS = 3
LEARNING_RATE = 2e-5
ATTRIBUTES = ['staff', 'cleanness', 'location']

# We use -1.0 to indicate a "missing label" in our dataset
MISSING_LABEL_VALUE = -1.0 

# ---------------------------------------------------------
# 2. CUSTOM DATASET
# ---------------------------------------------------------
class ReviewDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.attributes = ATTRIBUTES

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]
        review = item['review']
        labels = item['labels']

        # Tokenize the review
        encoding = self.tokenizer.encode_plus(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        # Prepare targets: Convert dictionary to a fixed-order list
        # If an attribute is missing in the data, we assign MISSING_LABEL_VALUE (-1)
        target_scores = []
        for attr in self.attributes:
            score = labels.get(attr, MISSING_LABEL_VALUE)
            # Optional: Normalize scores to 0-1 range if your raw data is 0-10
            # if score != MISSING_LABEL_VALUE: score = score / 10.0
            target_scores.append(score)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target_scores, dtype=torch.float)
        }

# ---------------------------------------------------------
# 3. THE MULTI-HEAD REGRESSION MODEL
# ---------------------------------------------------------
class MultiHeadAspectModel(nn.Module):
    def __init__(self, n_attributes):
        super(MultiHeadAspectModel, self).__init__()
        
        # Load the pre-trained BERT base (The "Shared Brain")
        self.bert = DistilBertModel.from_pretrained(MODEL_NAME)
        
        # Create a separate "Head" (Linear Layer) for each attribute
        # Each head takes the BERT hidden size (768) and outputs 1 number (score)
        self.heads = nn.ModuleList([
            nn.Linear(self.bert.config.hidden_size, 1) for _ in range(n_attributes)
        ])
        
        # Optional: Add a sigmoid activation if you normalized targets to 0-1
        # self.sigmoid = nn.Sigmoid() 

    def forward(self, input_ids, attention_mask):
        # Pass text through BERT
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Get the representation of the [CLS] token (sentence embedding)
        # Shape: (Batch_Size, 768)
        cls_output = output.last_hidden_state[:, 0, :]
        
        # Pass the same embedding to each specific head
        results = []
        for head in self.heads:
            # Result shape: (Batch_Size, 1) -> Flatten to (Batch_Size)
            head_output = head(cls_output).view(-1)
            results.append(head_output)
            
        # Stack results to shape (Batch_Size, n_attributes)
        return torch.stack(results, dim=1)

# ---------------------------------------------------------
# 4. TRAINING FUNCTION WITH MASKED LOSS
# ---------------------------------------------------------
def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0

    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)

        # Forward Pass
        predictions = model(input_ids=input_ids, attention_mask=attention_mask)

        # --- THE MAGIC TRICK: MASKED LOSS ---
        # Create a mask where True = Real Score exists, False = Missing (-1)
        mask = (targets != MISSING_LABEL_VALUE)
        
        # Filter out the missing values from both predictions and targets
        # We only calculate error on valid labels
        valid_preds = predictions[mask]
        valid_targets = targets[mask]

        if len(valid_targets) > 0:
            loss = loss_fn(valid_preds, valid_targets)
            
            # Backprop
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

    return total_loss / len(data_loader)

# ---------------------------------------------------------
# 5. EXECUTION
# ---------------------------------------------------------
if __name__ == "__main__":
    # Sample Data (Note: some attributes are missing)
    raw_data = [
        {
            "review": "The staff was incredibly helpful and kind.",
            "labels": {"staff": 10} # Missing cleanness/location
        },
        {
            "review": "Dirty room but the location is prime.",
            "labels": {"cleanness": 2, "location": 9} # Missing staff
        },
        {
            "review": "Average experience all around.",
            "labels": {"staff": 5, "cleanness": 5, "location": 5}
        }
    ]

    # Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
    
    # Dataset & Loader
    dataset = ReviewDataset(raw_data, tokenizer, MAX_LEN)
    data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    # Model Initialization
    model = MultiHeadAspectModel(n_attributes=len(ATTRIBUTES))
    model.to(device)

    # Optimization
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss() # Mean Squared Error for Regression

    # Training Loop
    print(f"Training on {device}...")
    for epoch in range(EPOCHS):
        loss = train_epoch(model, data_loader, loss_fn, optimizer, device)
        print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f}")

    # ---------------------------------------------------------
    # 6. INFERENCE (TESTING)
    # ---------------------------------------------------------
    print("\n--- Testing ---")
    test_review = "The location was terrible, heavily congested."
    
    model.eval()
    with torch.no_grad():
        inputs = tokenizer(test_review, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Get raw scores
        preds = model(inputs['input_ids'], inputs['attention_mask'])
        preds = preds.cpu().numpy()[0] # Convert to numpy array

        print(f"Review: '{test_review}'")
        for idx, attr in enumerate(ATTRIBUTES):
            # We clip the output to be safe, though regression can technically go outside bounds
            score = np.clip(preds[idx], 0, 10) 
            print(f"  {attr.capitalize()}: {score:.2f} / 10")